# Tu copia personal · Aula UTP

Este cuaderno se abre desde GitHub. En Colab elige **Archivo → Guardar una copia en Drive** antes de conservar cambios. Renombra esa copia y ejecuta de arriba abajo. Cada estudiante trabaja en su propio archivo y entorno.

Los ejemplos son didácticos: sigue la sección indicada por Cristian para esta clase. Las comprobaciones son voluntarias, sin entregas ni notas.

# Lectura guiada · Machine Learning UTP
Cristian Alvarado · Ejecuta de arriba hacia abajo. No hay tareas ni entregas. Los ejemplos se revisan con el docente.

## Q-learning: aprender una ruta en una simulación
GridWorld tiene inicio, paredes, pozo y meta. Sigue estado → acción → recompensa → nuevo estado. Un episodio es una trayectoria; la tabla Q estima valor de las acciones. El gráfico de recompensa y la política son resultados diferentes. No es una aplicación de riego ni un dron desplegado.

In [ ]:
from IPython.display import display
# ======================================================================
# Q-LEARNING TABULAR (Colab listo para pegar)
# Caso didáctico: GridWorld 5x5 con paredes, meta (recompensa +10) y pozo (-10)
# ======================================================================
# Objetivo:
# - Aprender una política π(s) que maximiza la recompensa acumulada.
# - Q-learning estima Q[s, a] (valor de acción) iterando:
#   Q[s,a] ← Q[s,a] + α * (r + γ * max_a' Q[s',a'] - Q[s,a])
#
# Conceptos clave:
# - Tabla Q: matriz estados×acciones con valores estimados.
# - Política ε-greedy: con prob. ε explora; con 1-ε explota el mejor Q.
# - Aprendizaje fuera de política (off-policy): usa el max_a' Q[s',a'].
# ======================================================================

import numpy as np
np.random.seed(42)
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------------------------------------
# 1) Definición del entorno GridWorld (discreto y tabular)
# --------------------------------------------------------
# Rejilla 5x5. Estados = celdas accesibles.
# Acciones: 0=Arriba, 1=Abajo, 2=Izquierda, 3=Derecha.
# Recompensas:
# - Paso normal: -1 (incentiva trayecto corto)
# - Meta (G): +10 y episodio termina
# - Pozo (X): -10 y episodio termina
# Paredes (#): no se puede entrar; quedarse en sitio penaliza -1
# Bordes: moverse fuera del tablero no está permitido; se ignora el movimiento y se penaliza -1.

H, W = 5, 5

# Mapa:
# S = inicio, G = meta, X = pozo, # = pared, . = libre
grid = np.array([
    list("S...."),
    list(".#..G"),
    list("..#.."),
    list("..X.."),
    list("....."),
])

start_pos = (0, 0)

# Utilidades de entorno
ACTIONS = [(-1,0), (1,0), (0,-1), (0,1)]  # U, D, L, R
N_ACTIONS = 4

def in_bounds(r, c):
    return 0 <= r < H and 0 <= c < W

def is_wall(r, c):
    return grid[r, c] == '#'

def to_state(r, c):
    return r * W + c

def from_state(s):
    return divmod(s, W)

def reset():
    return to_state(*start_pos)

def step(state, action):
    r, c = from_state(state)
    dr, dc = ACTIONS[action]
    nr, nc = r + dr, c + dc

    # Intento de movimiento
    if not in_bounds(nr, nc) or is_wall(nr, nc):
        nr, nc = r, c  # rebote
        reward = -1
        done = False
    else:
        cell = grid[nr, nc]
        if cell == 'G':
            reward = +10
            done = True
        elif cell == 'X':
            reward = -10
            done = True
        else:
            reward = -1
            done = False

    next_state = to_state(nr, nc)
    return next_state, reward, done

# Estados válidos (todas las celdas no pared)
mask_valid = (grid != '#')
N_STATES = H * W

# --------------------------------------------------------
# 2) Hiperparámetros de Q-learning
# --------------------------------------------------------
alpha = 0.1      # tasa de aprendizaje
gamma = 0.95     # factor de descuento
epsilon_start = 1.0
epsilon_end   = 0.05
epsilon_decay_episodes = 800  # episodios para decaer de start a end

episodes = 2000
max_steps = 200  # seguridad por episodio

# --------------------------------------------------------
# 3) Inicialización de la tabla Q
# --------------------------------------------------------
Q = np.zeros((N_STATES, N_ACTIONS), dtype=np.float32)

# Política ε-greedy
def epsilon_by_episode(ep):
    # Decaimiento lineal simple
    t = min(1.0, ep / epsilon_decay_episodes)
    return epsilon_start + t * (epsilon_end - epsilon_start)

def select_action(state, epsilon):
    if np.random.rand() < epsilon:
        return np.random.randint(N_ACTIONS)
    # Explota el mejor Q; desempata al azar
    qs = Q[state]
    max_q = np.max(qs)
    best_actions = np.flatnonzero(qs == max_q)
    return np.random.choice(best_actions)

# --------------------------------------------------------
# 4) Entrenamiento
# --------------------------------------------------------
returns = []

for ep in range(episodes):
    s = reset()
    total_reward = 0.0
    eps = epsilon_by_episode(ep)

    for t in range(max_steps):
        a = select_action(s, eps)
        s_next, r, done = step(s, a)

        # Regla central de Q-learning (off-policy)
        td_target = r + gamma * np.max(Q[s_next]) * (0 if done else 1)
        td_error  = td_target - Q[s, a]
        Q[s, a]  += alpha * td_error

        s = s_next
        total_reward += r
        if done:
            break

    returns.append(total_reward)

print("Entrenamiento completado.")
print("Recompensa media últimas 100 eps:", np.mean(returns[-100:]))

# --------------------------------------------------------
# 5) Visualización de aprendizaje
# --------------------------------------------------------
plt.figure(figsize=(7,3))
plt.plot(returns, linewidth=1)
plt.title("Recompensa por episodio")
plt.xlabel("Episodio")
plt.ylabel("Recompensa total")
plt.grid(True)
plt.show()

# --------------------------------------------------------
# 6) Extraer política óptima y mostrar tabla Q legible
# --------------------------------------------------------
# Política: acción con mayor Q por estado. Estados inválidos (#) se marcan.
arrow_map = {0:'^', 1:'v', 2:'<', 3:'>'}

policy_chars = grid.copy()
for r in range(H):
    for c in range(W):
        if grid[r,c] in ['#','G','X','S']:
            continue
        s = to_state(r,c)
        a = np.argmax(Q[s])
        policy_chars[r,c] = arrow_map[a]

print("Política óptima (flechas) sobre el mapa:")
for r in range(H):
    print(''.join(policy_chars[r]))

# Tabla Q como DataFrame (solo para estados no pared)
rows = []
for r in range(H):
    for c in range(W):
        if is_wall(r,c):
            continue
        s = to_state(r,c)
        rows.append({
            "estado": f"({r},{c})",
            "Q_up(^)":   round(float(Q[s,0]), 2),
            "Q_down(v)": round(float(Q[s,1]), 2),
            "Q_left(<)": round(float(Q[s,2]), 2),
            "Q_right(>)":round(float(Q[s,3]), 2),
            "celda": grid[r,c]
        })

q_df = pd.DataFrame(rows)
display(q_df.head(10))

# --------------------------------------------------------
# 7) Demostración: ejecutar un episodio con la política aprendida
# --------------------------------------------------------
def greedy_action(state):
    qs = Q[state]
    max_q = np.max(qs)
    best_actions = np.flatnonzero(qs == max_q)
    return np.random.choice(best_actions)

s = reset()
trajectory = [from_state(s)]
total = 0
for t in range(50):
    a = greedy_action(s)
    s, r, done = step(s, a)
    total += r
    trajectory.append(from_state(s))
    if done:
        break

print("Trayectoria con política aprendida:", trajectory)
print("Recompensa total de la demostración:", total)

# Visualización de la trayectoria
vis = grid.copy()
for (rr,cc) in trajectory:
    if vis[rr,cc] == '.':
        vis[rr,cc] = '*'
print("Mapa con trayectoria (*):")
for r in range(H):
    print(''.join(vis[r]))

# --------------------------------------------------------
# 8) Guía de lectura de resultados
# --------------------------------------------------------
print("""
================= CÓMO LEER LOS RESULTADOS =================
1) Curva de recompensa:
   - Tendencia ascendente y estabilización indican aprendizaje.
   - Oscilaciones son normales por la exploración ε-greedy.

2) Tabla Q:
   - Q[s,a] alto = esa acción en ese estado es prometedora.
   - En celdas cerca de 'G', verás Q positivos; cerca de 'X', negativos.

3) Política óptima (flechas):
   - Flechas muestran la acción greedy en cada celda libre.
   - Debe guiar desde 'S' a 'G' evitando 'X' y paredes.

4) Demostración:
   - 'Trayectoria' recorre celdas desde inicio hasta terminar.
   - Si termina en 'G' con pocos pasos y buena recompensa, la política es efectiva.

5) Parámetros:
   - α (learning rate) controla cuánto ajustas Q en cada paso.
   - γ (descuento) valora recompensas futuras.
   - ε (exploración) evita caer en óptimos locales; decae para explotar más al final.
=============================================================
""")
